# Caracal Bench CyberGym V4 - DEBUG MODE (1 task, sem base)

V3 travou em "Loading weights 47%" - hipotese: low_cpu_mem_usage off + verbose dup.

V4 fixes:
- `low_cpu_mem_usage=True` em load_model (evita pico RAM)
- `TRANSFORMERS_VERBOSITY=error` + `TRANSFORMERS_NO_ADVISORY_WARNINGS=1` (silencia verbose)
- `HF_HUB_DOWNLOAD_TIMEOUT=120` (fail fast em deadlock)
- Logger configurado com explicit `[load_model]` prints pra confirmar progresso
- **1 task** (`arvo:47101`) + `max_turns=3` + **skip base bench**

ETA V4: ~15-20min total. Se sucesso = expand pra 6 tasks + base em V5.

Settings: GPU T4 x2 (machine_shape NvidiaTeslaT4) + Internet ON + Persistence.

In [ ]:
BENCH = "cybergym"
CHECKPOINT_DATASET = "pedroafonso2/caracal-base-3b-s01"
OUTPUT_DATASET = "pedroafonso2/caracal-bench-cybergym-s01"
print(f"bench={BENCH} checkpoint={CHECKPOINT_DATASET}")

In [ ]:
!pip install -q 'transformers>=4.46.0' 'peft>=0.13.0' 'datasets>=3.0.0' kaggle 'requests>=2.31.0'

In [ ]:
import os
import subprocess

if not os.path.exists("/kaggle/working/caracal-1"):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            "dev",
            "https://github.com/iterate-labs-ai/caracal-1.git",
            "/kaggle/working/caracal-1",
        ],
        check=True,
    )
os.chdir("/kaggle/working/caracal-1")
rev = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"cloned, HEAD={rev}", flush=True)

In [ ]:
import torch

print(f"CUDA: {torch.cuda.is_available()}, n_gpu: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  gpu{i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import subprocess

ckpt_dir = "/kaggle/working/ckpt"
subprocess.run(
    [
        "kaggle",
        "datasets",
        "download",
        "-d",
        CHECKPOINT_DATASET,
        "-p",
        ckpt_dir,
        "--unzip",
        "--force",
    ],
    check=True,
)
subprocess.run(["ls", "-la", ckpt_dir], check=True)

In [ ]:
import os
import subprocess
import sys

os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"

MAX_TURNS = 3
TASKS = ["arvo:47101"]

subprocess.run(
    [
        sys.executable,
        "-u",
        "eval/run_cybergym_local.py",
        "--adapter",
        ckpt_dir,
        "--out",
        "/kaggle/working/bench-adapter-cybergym.json",
        "--max-turns",
        str(MAX_TURNS),
        "--cache-dir",
        "/kaggle/working/cybergym-cache",
        "--tasks",
        *TASKS,
    ],
    check=True,
    env={**os.environ},
)

In [ ]:
print("base bench skipped (debug v4 - rodando so adapter pra validar pipeline)")

In [ ]:
import json
from pathlib import Path

ad = json.loads(Path("/kaggle/working/bench-adapter-cybergym.json").read_text())
print(
    json.dumps(
        {
            "n_total": ad.get("n_total"),
            "n_pass": ad.get("n_pass"),
            "pass_at_1": ad.get("pass_at_1"),
            "max_turns": ad.get("max_turns"),
            "mythos_full_1507_baseline": 0.831,
        },
        indent=2,
    )
)
for r in ad.get("results", []):
    print(f"  task={r.get('task_id')} pass={r.get('pass')} turns={r.get('turns_used')}")
    for t in r.get("turns", []):
        print(
            f"    turn {t.get('turn')}: poc={t.get('poc_size_bytes')}B vul={t.get('vul_exit')} fix={t.get('fix_exit')} pass={t.get('pass')}"
        )

In [ ]:
print("publish skipped em V4 debug - rode V5 com 6 tasks + base depois de validar pipeline")